# Phase 6A - IndoBERTweet Prototype

Notebook ini menjalankan prototype **frozen IndoBERTweet** untuk smoke test awal. Encoder tidak di-fine-tune karena label saat ini baru 100 akun; classifier Logistic Regression dilatih pada embedding account-level hasil mean pooling postingan.

> Status hasil: `PROTOTYPE_ONLY`, bukan klaim performa final. Jalankan ulang setelah label baru ditambahkan.

In [1]:
from pathlib import Path
import json
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'PRD.md').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print('Repository:', ROOT)
print('Prototype outputs:', ROOT / 'output' / 'phase6_prototypes')


Repository: /home/andry-alfisi/Project/Project/Buzzer-Detection-
Prototype outputs: /home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase6_prototypes


In [2]:
import os
os.environ['HF_HUB_OFFLINE'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

from models.indobertweet_prototype import run_prototype

CHECKPOINT = 'indolem/indobertweet-base-uncased'
readiness = run_prototype(
    root=ROOT,
    checkpoint=CHECKPOINT,
    batch_size=8,
    max_length=128,
    test_size=0.30,
    device='cpu',
)
print(json.dumps(readiness, indent=2, ensure_ascii=False, default=str))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: indolem/indobertweet-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== IndoBERTweet PROTOTYPE ===
n: 30
accuracy: 0.8
precision_buzzer: 0.8125
recall_buzzer: 0.8125
f1_buzzer: 0.8125
roc_auc: 0.875
pr_auc: 0.872380795535304
confusion_matrix:
[[11  3]
 [ 3 13]]
{
  "status": "PROTOTYPE_ONLY",
  "model": "IndoBERTweet frozen encoder + account pooling + LogisticRegression",
  "generated_at_utc": "2026-11-11T09:49:55.180972+00:00",
  "repository": "/home/andry-alfisi/Project/Project/Buzzer-Detection-",
  "batches": [
    "batch2",
    "batch3"
  ],
  "metrics": {
    "n": 30,
    "accuracy": 0.8,
    "precision_buzzer": 0.8125,
    "recall_buzzer": 0.8125,
    "f1_buzzer": 0.8125,
    "confusion_matrix": [
      [
        11,
        3
      ],
      [
        3,
        13
      ]
    ],
    "roc_auc": 0.875,
    "pr_auc": 0.872380795535304
  },
  "artifacts": {
    "model": "/home/andry-alfisi/Project/Project/Buzzer-Detection-/output/phase6_prototypes/indobertweet/model/indobertweet_prototype.pkl",
    "embeddings": "/home/andry-alfisi/Project/Project/B

In [3]:
metrics = readiness['metrics']
assert readiness['status'] == 'PROTOTYPE_ONLY'
assert metrics['n'] > 0
assert Path(readiness['artifacts']['model']).exists()
assert Path(readiness['artifacts']['confusion_matrix']).exists()
print('IndoBERTweet notebook validation: OK')


IndoBERTweet notebook validation: OK


## Interpretasi

- Model menggunakan teks account-level gabungan dan mempertahankan hashtag, mention, emoji, serta marker URL dari kolom raw `text`.
- Evaluasi memakai split account-level 70/30 dengan seed tetap.
- Karena dataset kecil, angka metrik hanya untuk memastikan pipeline berjalan; jangan dipakai sebagai hasil final atau dasar vonis akun.